# Hybrid Pipeline — QAOA Stage (Quantum Ledger)

**Objective key:** `hybrid_qaoa` &nbsp;·&nbsp; **Family:** Hybrid &nbsp;·&nbsp; **Speed:** Compute-intensive

## Algorithm

Updated **Quantum Ledger 3-stage pipeline** — identical to `hybrid` but Stage 2 replaces
Simulated Annealing with a **QAOA gate-model circuit** over the same QUBO.

| Stage | Method | Backend |
|-------|--------|--------|
| 1 — IC Screening | Classical: rank μ/σ, keep top K_screen | CPU |
| 2 — QAOA Selection | QUBO → Ising → QAOA circuit → highest-prob K-hot bitstring | CPU (≤12) or IBM Runtime (≤20) |
| 3 — Markowitz Allocation | Convex Max-Sharpe on K_select chosen assets | CPU |

### Stage 2 QUBO → QAOA details

1. Build **Q** matrix (same formulation as `qubo_sa` and `hybrid`).
2. Map **Q** to **Ising (h, J)**: `h_i = -Q[i,i]/2 − Σ_{j≠i} Q[i,j]/2`, `J_ij = Q[i,j]/2`.
3. Run **p-layer QAOA** circuit: `|ψ₀⟩ = H^K|0…0⟩`, then alternate cost `U_C(γ)` + mixer `U_B(β)` layers.
4. COBYLA minimises `⟨ψ|H_C|ψ⟩` over angles **(γ, β)**.
5. Extract: **highest-probability K_select-hot bitstring** → equal weight into Markowitz.

### IBM Quantum Runtime path

When `execution_kind: ibm_runtime` and IBM token is configured:
- Stage 2 uses **EstimatorV2** (energy expectation) for angle optimisation + **SamplerV2** for final bitstring.
- Stages 1 and 3 remain on CPU.
- K_screen cap: **20 qubits** (same as `MAX_IBM_QUBITS`).

## Papers

- **Foundational (stage 2):** Farhi, Goldstone & Gutmann (2014) — *A Quantum Approximate Optimization Algorithm*  
  https://arxiv.org/abs/1411.4028 &nbsp;·&nbsp; PDF: https://arxiv.org/pdf/1411.4028
- **Foundational (stage 3):** Markowitz (1952) — *Portfolio Selection* — Journal of Finance  
  https://doi.org/10.1111/j.1540-6261.1952.tb01525.x
- **Modern (related reading):** Zhou et al. (2020) — *QAOA: Performance, Mechanism, and Implementation*  
  https://arxiv.org/abs/1812.01041 &nbsp;·&nbsp; PDF: https://arxiv.org/pdf/1812.01041

## Assumptions

- `mu` and `Sigma` are annualised.
- `K_screen ≤ 12` for classical path (statevector cap). Set `K_screen` explicitly if your universe is larger.
- `p=2`, `n_qaoa_restarts=2`, `seed=42`.
- All weights from Stage 3 satisfy the `(weight_min, weight_max)` bounds.

In [ ]:
import sys
from pathlib import Path

def _find_root() -> Path:
    for p in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]:
        if (p / "api" / "app.py").is_file():
            return p
    raise RuntimeError("Cannot find repo root — run from notebooks/objectives/ or repo root")

ROOT = _find_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f"Repo root: {ROOT}")

In [ ]:
import numpy as np
from services.portfolio_optimizer import run_optimization

rng = np.random.default_rng(42)
n = 15  # full universe
# K_screen will auto-cap at 12 for classical QAOA path
mu = rng.uniform(0.06, 0.22, n)
cov_raw = rng.standard_normal((n, n))
Sigma = cov_raw @ cov_raw.T / n + np.eye(n) * 0.04
asset_names = [f"ASSET_{i+1:02d}" for i in range(n)]
print(f"Universe: {n} assets")
print("mu range:", mu.min().round(4), "—", mu.max().round(4))

In [ ]:
# Classical path: K_screen=8 (≤12), K_select=4
# Stage 2 uses numpy QAOA statevector (no IBM token needed)
result = run_optimization(
    returns=mu,
    covariance=Sigma,
    objective="hybrid_qaoa",
    asset_names=asset_names,
    K_screen=8,
    K_select=4,
    lambda_risk=1.0,
    gamma=8.0,
    n_restarts=2,   # maps to n_qaoa_restarts
    weight_min=0.005,
    weight_max=0.30,
    seed=42,
)

active = [(name, w) for name, w in zip(asset_names, result.weights) if w > 1e-6]
print(f"\nObjective:       {result.objective}")
print(f"Selected assets: {len(active)} (K_select=4)")
print(f"Expected return: {result.expected_return:.4f}")
print(f"Volatility:      {result.volatility:.4f}")
print(f"Sharpe ratio:    {result.sharpe_ratio:.4f}")
print("\nFinal portfolio (Stage 3 Markowitz):")
for name, w in active:
    print(f"  {name}: {w:.4f}")

In [ ]:
# Stage diagnostics
if hasattr(result, 'stage_info') and result.stage_info:
    info = result.stage_info
    print("Stage diagnostics:")
    print(f"  Stage 1 screened count: {info.get('stage1_screened_count', 'n/a')}")
    print(f"  Stage 2 solver:         {info.get('stage2_solver', 'qaoa')}")
    print(f"  Stage 2 selected:       {info.get('stage2_selected_names', 'n/a')}")
    print(f"  Stage 2 QUBO energy:    {info.get('stage2_qubo_obj', 'n/a')}")
    print(f"  Stage 3 Sharpe:         {info.get('stage3_sharpe', 'n/a')}")

In [ ]:
# Compare: hybrid (SA stage 2) vs hybrid_qaoa (QAOA stage 2)
hybrid_sa = run_optimization(
    returns=mu, covariance=Sigma, objective="hybrid",
    asset_names=asset_names, K_screen=8, K_select=4,
    lambda_risk=1.0, gamma=8.0, weight_min=0.005, weight_max=0.30, seed=42,
)
ew = run_optimization(returns=mu, covariance=Sigma, objective="equal_weight")

print("Sharpe comparison (same universe, K_screen=8, K_select=4):")
print(f"  hybrid_qaoa (QAOA stage 2): {result.sharpe_ratio:.4f}")
print(f"  hybrid (SA stage 2):        {hybrid_sa.sharpe_ratio:.4f}")
print(f"  Equal Weight:               {ew.sharpe_ratio:.4f}")
print("\nNote: QAOA and SA solve the same QUBO combinatorial problem;")
print("results may differ due to solver stochasticity and angle initialisation.")

In [ ]:
# IBM Quantum Runtime path (informational)
print("IBM Quantum Runtime path for hybrid_qaoa:")
print("  Requires: configured IBM token (POST /api/config/ibm-quantum)")
print("  Payload:  execution_kind='ibm_runtime', objective='hybrid_qaoa'")
print("  Stage 1:  IC screening (CPU, unchanged)")
print("  Stage 2:  QAOA on IBM Runtime — EstimatorV2 + SamplerV2")
print("            K_screen cap: 20 qubits (MAX_IBM_QUBITS)")
print("  Stage 3:  Markowitz Max-Sharpe (CPU, unchanged)")
print("\nClassical statevector path:")
print("  K_screen cap: 12 qubits (MAX_SIM_QUBITS — statevector 2^K_screen)")
print("  Use objective='hybrid' (SA) for larger universes without IBM token")